<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "591"
iso_country = "Panama"
import_portfolio = "h6_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2025
gdp = 90.460e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Panama': 

Number of trade records present in dataset: 23570
Number of partner countries present in dataset: 100
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$7732.8m
Total weight within the dataset: 4862.4 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,591,Panama,724,Spain,Import,Sea,761610,"761610 | Aluminium; nails, ta",7616,7616 | Aluminium; articles,76,76 | Aluminium and articl,42944,"Nails, tacks, staples (except staples in strip...",Metal Products,37,FMP - Metal products,23139,NaN,23139,NaN,0.0,2404,2316.2139,7753.0,4817.489363,11158.335826,['Container'],['Container'],30.812793,30.721833,30.731084,30.731084,30.777030,30.777030,32.432358,60.475665,35.296424,33.038212,33.171271,34.111148,31.670630,58.991897,37.557983,34.405465,33.487816,35.284056,31.002141,65.836522,46.149515,41.086201,37.788055,36.918317,30.345698,68.780204,55.244500,51.092081,40.554846,39.822333,29.701952,70.945427,69.540120,68.431495,43.614441,41.970641,29.045546,69.974657,66.742093,72.660045,56.223976,42.765552,28.407229,60.363684,65.304967,71.854015,53.416025,43.278518,27.777826,54.183232,55.170817,65.826310,53.291500,43.612716,42944
1,1,591,Panama,724,Spain,Import,Sea,820320,"820320 | Tools, hand; pliers",8203,"8203 | Tools, hand; files,",82,"82 | Tools, implements, c",42921,Hand tools (including hand tools of a kind use...,Metal Products,37,FMP - Metal products,6445,NaN,6445,NaN,0.0,870,552.9810,7753.0,4817.489363,2663.980085,['Container'],['Container'],7.356354,7.334638,7.336846,7.336846,7.347816,7.347816,7.743015,14.438172,8.426792,7.887658,7.919425,8.143815,7.561157,14.083932,8.966724,8.214081,7.994998,8.423839,7.401560,15.718041,11.017897,9.809063,9.021652,8.814008,7.244838,16.420826,13.189265,12.197902,9.682206,9.507323,7.091148,16.937759,16.602251,16.337574,10.412664,10.020217,6.934435,16.705994,15.934240,17.347113,13.423109,10.209997,6.782041,14.411437,15.591136,17.154679,12.752729,10.332465,6.631775,12.935894,13.171674,15.715603,12.722999,10.412252,42921


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $1952.8m (25.3%), ... 

                            cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
10 | Cereals               245041978  901005242.1      3.2     18.5
23 | Food industries, res  222941188  323978341.0      2.9      6.7
21 | Miscellaneous edible  176328395   48629885.2      2.3      1.0
04 | Dairy produce; birds  164745758   44297264.7      2.1      0.9
22 | Beverages, spirits a  163274031  805951264.9      2.1     16.6
19 | Preparations of cere  161048934   56910586.0      2.1      1.2
20 | Preparations of vege  141011826   94038555.3      1.8      1.9
02 | Meat and edible meat  136020344   52309667.2      1.8      1.1
15 | Animal, vegetable or  107068595   92145545.4      1.4      1.9
16 | Meat, fish, crustace  107029987   23416233.9      1.4      0.5
08 | Fruit and nuts, edib   74643022   54108985.9      1.0      1.1
03 | Fish and 

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $1746.3m (22.6%), ... 

                   cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                  
Food & Beverages  999096065  1.333296e+09     12.9     27.4
Agriculture       740586618  1.038235e+09      9.6     21.4
Fishing             6609084  1.051833e+06      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $1610.0m (20.8%), ... 

                                cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     743673507  363886695.2      9.6      7.5
VOL - Vegetable oils and fats  188380510  302982594.0      2.4      6.2
OMT - Meat products n.e.c.     158332500   50121266.8      2.0      1.0
MIL - Dairy products           157713193   37346030.9      2.0      0.8
GRO - Cereal grains n.e.c.     152370685  629352732.1      2.0     12.9
V_F - Vegetables, fruit, nuts   94742449   97353424.0      1.2      2.0
CMT - Bovine meat prods         46527806   13409585.4      0.6      0.3
OCR - Crops n.e.c.              28136499   24259248.0      0.4      0.5
PDR - Paddy rice                22395744   38901407.3      0.3      0.8
PCR - Processed rice            11166121   19395552.2      0.1      0.4
OAP - Animal products n.e.c.     3836868    2900393.0      0.0      0

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 25.3% 
GTAP 20.8% 
EORA 22.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_16221/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,591,Panama,724,Spain,Import,Sea,761610,"761610 | Aluminium; nails, ta",7616,7616 | Aluminium; articles,76,76 | Aluminium and articl,42944,"Nails, tacks, staples (except staples in strip...",Metal Products,37,FMP - Metal products,23139,NaN,23139,NaN,0.0,2404,2316.2139,7753.0,4817.489363,11158.335826,['Container'],['Container'],30.812793,30.721833,30.731084,30.731084,30.777030,30.777030,32.432358,60.475665,35.296424,33.038212,33.171271,34.111148,31.670630,58.991897,37.557983,34.405465,33.487816,35.284056,31.002141,65.836522,46.149515,41.086201,37.788055,36.918317,30.345698,68.780204,55.244500,51.092081,40.554846,39.822333,29.701952,70.945427,69.540120,68.431495,43.614441,41.970641,29.045546,69.974657,66.742093,72.660045,56.223976,42.765552,28.407229,60.363684,65.304967,71.854015,53.416025,43.278518,27.777826,54.183232,55.170817,65.826310,53.291500,43.612716,42944,NaN
1,1,591,Panama,724,Spain,Import,Sea,820320,"820320 | Tools, hand; pliers",8203,"8203 | Tools, hand; files,",82,"82 | Tools, implements, c",42921,Hand tools (including hand tools of a kind use...,Metal Products,37,FMP - Metal products,6445,NaN,6445,NaN,0.0,870,552.9810,7753.0,4817.489363,2663.980085,['Container'],['Container'],7.356354,7.334638,7.336846,7.336846,7.347816,7.347816,7.743015,14.438172,8.426792,7.887658,7.919425,8.143815,7.561157,14.083932,8.966724,8.214081,7.994998,8.423839,7.401560,15.718041,11.017897,9.809063,9.021652,8.814008,7.244838,16.420826,13.189265,12.197902,9.682206,9.507323,7.091148,16.937759,16.602251,16.337574,10.412664,10.020217,6.934435,16.705994,15.934240,17.347113,13.423109,10.209997,6.782041,14.411437,15.591136,17.154679,12.752729,10.332465,6.631775,12.935894,13.171674,15.715603,12.722999,10.412252,42921,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $1144.6m of $7732.8m (14.8%)
Volume: 1370.4 of 4862.4 million kilograms (28.2%)
Value-to-Weight: 0.84 from 1.59 US$ per kg (+-47.5%)
Average distance to market (volume-weighted): 2438.4 from 3794.9 nm (+-35.7%)

            food_group  N_com   cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    638  430347422  223078295.8      5.6      4.6
1  Unprocessed Cereals     28  211480113  842708282.6      2.7     17.3
2                Dairy    109  161727830   43291358.6      2.1      0.9
3        Fats and Oils    117   96414385   79913044.0      1.2      1.6
4        Meat and Prep     41   76022316   36697145.6      1.0      0.8
5           Vegetables    297   71738974   81287168.5      0.9      1.7
6               Fruits    169   68819413   51502062.2      0.9      1.1
7      Sugar and Honey     87   28050730   11920445.7      0.4      0.2



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,43650828.8,78759013.6,47233567.8,44337160.1,44496596.1,45765495.3,40240397.6,91965593.8,89793102.8,89266616.5,57757388.4,55650133.2,37824638.8,71507158.0,72946340.1,86119860.8,70155516.4,57777716.6
1,Fats and Oils,125543343.1,203394515.3,133621731.0,125475748.3,125829088.4,130243934.2,117977415.9,227868773.0,224890363.2,232984147.6,159660768.2,151924715.6,112686716.7,190542247.6,192536647.8,222013391.6,186329413.3,156488713.6
2,Fruits,37569576.0,69341804.5,40806012.5,38232591.9,38382359.0,39475010.3,34474667.8,81391396.1,79586862.5,78562877.7,50314847.5,48393323.6,32292676.3,62433398.6,63603236.7,75601705.0,61332308.9,50268483.7
3,Meat and Prep,19543658.8,36442486.0,21269537.8,19908744.9,19988926.0,20555293.4,17898322.7,42751538.6,41904704.1,41236649.3,26281954.1,25291404.2,16738849.0,32650681.4,33245797.9,39666771.5,32113326.0,26280914.4
4,Processed Cereals,214332949.2,380265133.7,230824882.7,216920350.8,217728252.9,224282743.7,198151085.0,448577374.5,436273144.6,435395404.3,285443306.0,270683556.2,186754596.8,349291928.7,356062591.2,418204070.9,342862352.9,280340757.7
5,Sugar and Honey,10738158.8,18813506.8,11546060.1,10862230.4,10900467.3,11226358.1,9949439.4,22215460.9,21445104.7,21530196.5,14106752.8,13510703.1,9391573.4,17326410.4,17722935.6,20661837.7,17018667.5,14005279.1
6,Unprocessed Cereals,452643170.9,802118044.9,487211701.7,457881519.9,459603614.2,473558536.3,418562286.0,946883856.3,921941368.5,919783514.4,604663017.8,570976537.8,394590186.4,737985982.0,751727660.0,883273092.7,724277919.2,590993502.7
7,Vegetables,94463349.3,171963245.5,102325174.8,95960857.9,96327862.2,99113395.4,86922518.7,202088483.5,196930881.1,195328888.8,125900271.6,120897082.2,81582255.6,155692328.2,158778691.1,187886406.4,152990436.3,125498014.6


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.22,0.02,0.00,0.01,0.01,0.32,0.31,0.30,0.11,0.10,0.21,0.22,0.30,0.20,0.12
1,Fats and Oils,0.81,0.08,-0.00,0.00,0.05,1.14,1.11,1.19,0.43,0.35,0.81,0.83,1.13,0.76,0.45
2,Fruits,0.46,0.05,0.01,0.01,0.03,0.68,0.66,0.64,0.23,0.20,0.44,0.45,0.63,0.42,0.26
3,Meat and Prep,0.22,0.02,0.00,0.01,0.01,0.33,0.32,0.31,0.11,0.10,0.21,0.22,0.30,0.20,0.13
4,Processed Cereals,0.39,0.04,0.01,0.01,0.02,0.58,0.55,0.55,0.20,0.17,0.38,0.39,0.54,0.36,0.22
5,Sugar and Honey,0.29,0.03,0.00,0.01,0.02,0.44,0.41,0.41,0.15,0.13,0.28,0.30,0.40,0.27,0.16
6,Unprocessed Cereals,1.65,0.16,0.02,0.03,0.10,2.50,2.38,2.37,0.88,0.72,1.62,1.69,2.31,1.56,0.93
7,Vegetables,1.08,0.11,0.02,0.03,0.06,1.61,1.53,1.51,0.54,0.47,1.03,1.08,1.48,1.00,0.61
